### 多角色聊天机器人流水线（编码挑战）

在此笔记本中实现一条「多模型接力」流水线：先给定一道编码挑战，再由不同角色模型轮流推进。

这对应 Week 2 的多 chatbot / 角色分工思路：同一挑战在多个 system prompt 下被拆成提示 → 规划 → 编码 → 审查 → 重构。

##### 模型 1：提示器 Peter（Prompter）

提出并细化发给其他模型的 prompt。

##### 模型 2：规划器 Paul（Planner）

给出 3 条解题思路，并推荐最佳做法。

##### 模型 3：编码器 Cody（Coder）

根据计划写出代码。

##### 模型 4：审稿人 Ren（Reviewer）

审查代码，并向用户总结方案；若需重构会在回复里显式写出 `refactor the code`。

##### 模型 5：重构 Rob（Refactor）

根据审查意见重构代码。

## 怎么跑

1. `.env` 里配置 `OPENROUTER_API_KEY`（需以 `sk-or-v1` 开头）
2. 同目录准备 `challenge.md` 作为挑战题输入
3. 从上到下运行；结束会写出 `conversation.md`


In [ ]:
# ========== 设置环境：读 OpenRouter 密钥 ==========
# 导入标准库 os：读环境变量
import os

# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown、display：笔记本展示
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：经 OpenRouter 调多家模型
from openai import OpenAI

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# OpenRouter API Key（名字必须是 OPENROUTER_API_KEY）
api_key: str | None = os.getenv("OPENROUTER_API_KEY")
# OpenRouter 的 OpenAI 兼容 base_url（URL 原样保留）
base_url: str = "https://openrouter.ai/api/v1"

# 断言密钥前缀：不是 sk-or-v1 就立刻失败，避免静默用错 key
assert api_key[:8] == "sk-or-v1", "OpenRouter API Key must start with sk-or-v1"


In [ ]:
# ========== 设置不同角色函数：各自 system prompt + 不同 OpenRouter 模型 ==========
# 从 openai 导入 Stream：流式响应的类型标注
from openai import Stream


# 指向 OpenRouter 的客户端（一家网关，多家后端模型）
openrouter = OpenAI(api_key=api_key, base_url=base_url)

# 五个角色名列表：拼 user prompt 时用来列出「你在和谁对话」
models: list[str] = ["Peter", "Paul", "Cody", "Ren", "Rob"]


def get_user_prompt(conversation: str, name: str) -> str:
    """获取对话的用户提示：把历史与角色名填进英文 user 模板。"""
    # 其他人名 = 角色列表去掉自己
    other_models_list: list[str] = list(filter(lambda x: x != name, models))
    # 拼成 "A, B and C" 这种英文列举，供 prompt 使用
    other_models: str = (
        ", ".join(other_models_list[:-1]) + " and " + other_models_list[-1]
    )
    # 返回英文 user prompt（模板原文不可改译）
    return f"""
    You are {name}, in conversation with {other_models}.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as {name}.
    """


def prompter(converstation: str) -> Stream:
    # Peter：提示工程师；参数名 converstation 保持原拼写（勿重命名）
    # system prompt 英文原样：规定 Peter 如何写给其他模型的任务说明
    system_prompt = """
    You are Peter who is a senior software engineer. 
    You are responsible for coming up with the prompt for the other models. You will be given an initial 
    prompt and you will come up with clear, concise and specific prompts for the other models. You will ensure that the other
    models understand the task and the constraints. You will also ensure that there are no regressions in as the tasks you will be
    prompting for are coding challenges.
    
    The other models are:
    - Paul: who is good at coming up with plans
    - Cody: who is good at coding
    - Ren: who is good at reviewing code
    - Rob: who is good at refactoring code
    """

    # 流式调用 OpenRouter 上的 openai/gpt-5.1-chat
    stream = openrouter.chat.completions.create(
        model="openai/gpt-5.1-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(converstation, "Peter")},
        ],
        # stream=True：边生成边返回 chunk
        stream=True,
    )
    # 把 stream 交给外层去逐块展示
    return stream


def planner(conversation: str) -> Stream:
    # Paul：规划者；给出 3 方案 + 推荐
    # system prompt 英文原样保留
    system_prompt = """
    You are Paul who is a senior software engineer. 
    You are responsible for coming up with the plan for the Cody who is a mid level software engineer.
    You will give three suggestions of how to solve the task. You will then give a recommendation of the best approach.
    You will come up with clear, concise and specific plans for the Cody.
    You will ensure that the Cody understands the task and the constraints.
    You will also ensure that there are no regressions in as the tasks you will be planning for are coding challenges.

    The other models are:
    - Peter: who is good at coming up with prompts
    - Cody: who is good at coding
    - Ren: who is good at reviewing code
    - Rob: who is good at refactoring code
    """
    # 流式调用 google/gemini-3-pro-preview
    stream = openrouter.chat.completions.create(
        model="google/gemini-3-pro-preview",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(conversation, "Paul")},
        ],
        stream=True,
    )
    return stream


def coder(conversation: str) -> Stream:
    # Cody：根据 Paul 的计划写代码
    # system prompt 英文原样保留
    system_prompt = """
    You are Cody who a mid level software engineer. 
    You are responsible for coding the solution to the task. You will be given a plan by Paul and you will code the solution.
    You will ensure that the solution is correct and that it is within the constraints.

    The other models are:
    - Paul: who is good at coming up with plans
    - Peter: who is good at coming up with prompts
    - Ren: who is good at reviewing code
    - Rob: who is good at refactoring code
    """

    # 流式调用 anthropic/claude-sonnet-4.6（Coder）
    stream = openrouter.chat.completions.create(
        model="anthropic/claude-sonnet-4.6",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(conversation, "Cody")},
        ],
        stream=True,
    )
    return stream


def reviewer(conversation: str) -> Stream:
    # Ren：QA；若需重构必须在回复中出现短语 refactor the code；通过则 looks good to me
    # （这两句英文触发词是控制流依赖，绝对不能改）
    system_prompt = """
    You are Ren who is a senior QA engineer.
    You are responsible for reviewing all the code.
    You will be given the code and you will review it.
    You will ensure that the code is correct and that it is within the constraints.
    Make sure that the code is DRY, SOLID and follows all the best practices.

    If code needs to refactored, make sure you explicitly add the phrase `refactor the code` to your the response.
    If the code looks good, make sure you explicitly add the phrase `looks good to me` to the response.

    The other models are:
    - Peter: who is good at coming up with prompts
    - Paul: who is good at coming up with plans
    - Cody: who is good at coding
    - Rob: who is good at refactoring code
    """

    # 流式调用 z-ai/glm-5
    stream = openrouter.chat.completions.create(
        model="z-ai/glm-5",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(conversation, "Ren")},
        ],
        stream=True,
    )
    return stream


def refactor(conversation: str) -> Stream:
    # Rob：按审查意见重构 Cody 的代码
    # system prompt 英文原样保留
    system_prompt = """
    You are Rob who is a principal engineer.
    You are responsible for refactoring the code of Cody. You will be given the code and you will refactor it.
    You will ensure that the code is correct and that it is within the constraints.

    The other models are:
    - Paul: who is good at coming up with plans
    - Cody: who is good at coding
    - Ren: who is good at reviewing code
    - Peter: who is good at coming up with prompts
    """

    # 流式调用 anthropic/claude-sonnet-4.6（Refactor）
    stream = openrouter.chat.completions.create(
        model="anthropic/claude-sonnet-4.6",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(conversation, "Rob")},
        ],
        stream=True,
    )
    return stream



In [ ]:
# ========== 开始对话：流式展示 + 审查/重构循环 ==========
# NoReturn：标注函数理论上不按正常值返回（这里仍会跑完写文件）
from typing import NoReturn

# DisplayHandle / update_display：同一展示位上刷新流式 Markdown
from IPython.display import DisplayHandle, update_display


def display_stream_response(response: Stream, display_handle: DisplayHandle):
    # 把流式 chunk 拼成完整字符串，并边到边刷新同一个 display
    response_stream = ""
    for chunk in response:
        # delta.content 可能为 None（例如结束块），用 or "" 护住
        response_stream += chunk.choices[0].delta.content or ""
        update_display(Markdown(response_stream), display_id=display_handle.display_id)
    return response_stream


def begin_converstation(challenge: str) -> NoReturn:
    # 注意函数名 begin_converstation 保持原拼写；challenge 来自 challenge.md
    # 初始 conversation 文本：嵌入用户挑战（中间英文/标题结构保持原样）
    conversation = f"""
    Initial prompt:

    # # 彼得
     
        Here is the user challenge

    ``` txt
    {challenge}
    ```

    """
    # 创建一个可就地更新的空 Markdown 显示位（display_id=True）
    display_handle = display(Markdown(""), display_id=True)

    # 1) Peter 细化 prompt，并流式显示
    prompter_response = display_stream_response(prompter(conversation), display_handle)

    # 2) 把 Peter 回复追加进历史，再让 Paul 做规划
    conversation += f"## Peter:\n\n{prompter_response}\n"
    planner_response = display_stream_response(
        planner(prompter_response), display_handle
    )

    # 3) Paul 之后交给 Cody 编码
    conversation += f"## Paul:\n\n{planner_response}\n"
    coder_response = display_stream_response(coder(planner_response), display_handle)

    # 4) Cody → Ren 审查 → Rob 重构 → Ren 再审
    conversation += f"## Cody:\n\n{coder_response}\n"
    reviewer_response = display_stream_response(
        reviewer(coder_response), display_handle
    )
    conversation += f"## Ren:\n\n{reviewer_response}\n"
    refactor_response = display_stream_response(
        refactor(reviewer_response), display_handle
    )
    conversation += f"## Rob:\n\n{refactor_response}\n"
    reviewer_response = display_stream_response(
        reviewer(refactor_response), display_handle
    )

    # 最多再循环若干次；用审查回复里的英文短语决定分支
    refactor_count = 1
    while refactor_count < 5:
        # 审查要求重构：偶数轮找 Rob，奇数轮再让 Cody 改
        if "refactor the code" in reviewer_response:
            if refactor_count % 2 == 0:
                conversation += f"## Ren:\n\n{reviewer_response}\n"
                refactor_response = display_stream_response(
                    refactor(reviewer_response), display_handle
                )
                conversation += f"## Rob:\n\n{refactor_response}\n"
                reviewer_response = display_stream_response(
                    reviewer(refactor_response), display_handle
                )
            else:
                conversation += f"## Ren:\n\n{reviewer_response}\n"
                coder_response = display_stream_response(
                    coder(reviewer_response), display_handle
                )
                conversation += f"## Cody:\n\n{coder_response}\n"
                reviewer_response = display_stream_response(
                    reviewer(coder_response), display_handle
                )
        # 审查通过：结束循环
        elif "looks good to me" in reviewer_response:
            break

        # 轮次 +1，防止无限来回重构
        refactor_count += 1

    # 把最终审查意见标成 FINAL RESPONSE，并写入 conversation 文本
    conversation += f"### FINAL RESPONSE\n\n{reviewer_response}\n"

    # 整段对话落盘，便于课后回看
    with open("conversation.md", "w", encoding="utf-8") as f:
        f.write(conversation)

    # 最后在笔记本里再展示完整 Markdown
    display(Markdown(conversation))



In [ ]:
# ========== 入口：读挑战题并启动流水线 ==========
# 从同目录 challenge.md 读取挑战题正文（UTF-8）
with open("challenge.md", "r", encoding="utf-8") as f:
    challenge = f.read()

# 启动多角色对话（函数名拼写保持原样）
begin_converstation(challenge)
